# 🚢 Titanic dataset EDA 🔍

###     [СССЫЛКА НА ДАТАСЕТ](https://www.kaggle.com/competitions/titanic/data)

Классический ML-датасет для предсказания выживаемости(survived = 0/1) пассажиров на Титанике

| Колонка   | Описание                                             | Тип               |
|-----------|------------------------------------------------------|-------------------|
| Survived  | Целевая переменная (0 = погиб, 1 = выжил)            | Бинарный          |
| Pclass    | Класс билета (1, 2, 3)                               | Порядковый        |
| Name      | Имя пассажира                                        | Текст             |
| Sex       | Пол (male/female)                                    | Категориальный    |
| Age       | Возраст                                              | Числовой          |
| SibSp     | Кол-во братьев/супругов на борту                     | Числовой          |
| Parch     | Кол-во родителей/детей на борту                      | Числовой          |
| Ticket    | Номер билета                                         | Текст             |
| Fare      | Цена билета                                          | Числовой          |
| Cabin     | Номер каюты                                          | Категориальный    |
| Embarked  | Порт посадки (C, Q, S)                               | Категориальный    |

### Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

### Загрузка данных

In [ ]:
df = pd.read_csv('train.csv')

## Быстрый обзор данных(pandas) 🐼🐼🐼

In [ ]:
for item in df.columns: # Названия столбцов
    print(item)

count = df.columns.size
print(f"\nCount: {count}")

In [ ]:
df.head() # вывод первых строк(5 по умолчанию)

In [ ]:
df.tail() # вывод последних строк(5 по умолчанию)

In [ ]:
df.shape # размерность датасета

In [ ]:
df.info() # Краткая сводка о датасете(Названия столбцов, количество непустых значений в столбцах, типы данных столбцов)

In [ ]:
df.describe().round() # Статистическое описание

In [ ]:
df.describe(include='str')

# Что видим:
# count  - количество непустых значений
# unique - количество уникальных значений
# top    - самое частое значение (мода)
# freq   - частота самого частого значения

In [ ]:
df.duplicated().sum() # Проверка на полные дубликаты строк

Дубликатов нет

## 🧩 Обработка пропусков

In [28]:
df.isnull().sum() # Количество пропусков по столбцам

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

Что мы видим: В Cabin много пропусков(687), а в Emarked всего 2(<1%), сразу напрашивается желание просто выкинуть из датасета двух пассажиров без Embarked, т.к. таких пассажиров очень мало, но мы заполним их модой, чтобы по минимуму терять данные. а столбец Cabin удалить, так как опасно заполнять такое огромное количество пропусков "искуственными" данными

In [ ]:
# df = df.dropna(subset='Embarked') # Удалим строки, где Embarked отсутствует МОЖНО ТАК

df = df.dropna(axis=1, thresh=len(df) * 0.5) # удалим столбцы, где >50% пропусков(axis - флаг, отвечающий за удаление строк или столбцов, thresh - по сути порог, если он превышен - удаление)

In [24]:
# Подсчёт моды для категориального признака Embarked(порт высадки)



mode_embarked = df['Embarked'].mode()[0]

print(mode_embarked)

S


In [27]:
# Заполним пропуски модой

df['Embarked'] = df['Embarked'].fillna(mode_embarked)

С признаком Age несколько интереснее, 

In [ ]:
df.isnull().sum() # Проверка пропусков после удаления

In [ ]:
# Построим boxplot для выбора дальнейшей стратегии заполнения пустых Age
fig2 = px.box(df, y='Age')


fig2.show()


По графику видно, что есть возраста выше upperfence = 65, но это **НЕ ВЫБРОСЫ**, тут нет никакой аномалии, это естественно для возраста людей, на борту находились пожилые люди.


### Гипотезы для дальнейшего исследования
Возраст может коррелировать с другими признаками:
- **Класс билета (pclass)** — возможно, в 1-м классе пассажиры старше
- **Цена билета (fare)** — дорогие билеты могли покупать взрослые/пожилые
- **Пол (sex)** — возможно, разный возраст у мужчин и женщин

In [ ]:
plt.figure(figsize=(15, 5))

# Гипотеза 1: зависимость от класса
plt.subplot(1, 3, 1)
sns.boxplot(x='Pclass', y='Age', data=df)
plt.title('Возраст по классам')

# Гипотеза 2: зависимость от цены
plt.subplot(1, 3, 2)
sns.scatterplot(x='Fare', y='Age', hue='Pclass', data=df, alpha=0.8)
plt.title('Цена билета в соответствии с возрастом')

# Гипотеза 3: зависимость от пола
plt.subplot(1, 3, 3)
sns.boxplot(x='Sex', y='Age', data=df)
plt.title('Возраст по полу')

plt.show()

**Что мы видим:** в 1 классе явно больше взрослых/стариков, а в промежутке от 20 до 60 лет больше всего дорогих билетов(>100). А от пола возраст не зависит, что вполне ожидаемо.
более явная статистика указана в первом случае(возраст коррелирует с классом билета)  

**Дальнешая стратегия заполнения:** Итак, на основание нашей проверки гипотезы **будем заполнять пропуски в Age медианой по классу билетов**, так как **глобальная медиана не подойдёт**, потому что она будет игнорировать, что в 1 классе люди зачастую старше, так же отмечу, что **заполнение средним не подойдет** так как есть пожилые люди, и среднее чувствительно к их значениями. **Заполнение модой бессмысленно**, т.к. много уникальных значений в возрасте.

In [ ]:
# Считаем медианы по классам
medians = df.groupby('Pclass')['Age'].median()

for pclass, median in medians.items():
    print(f"  Класс {pclass}: {median:.1f} лет")

In [ ]:
# Заполняем медианой пропуски в Age в соответствии с классом билета

for pclass, median in medians.items():
    mask = (df['Pclass'] == pclass) # булева маска

    df.loc[mask, 'Age'] = df.loc[mask, 'Age'].fillna(median) # выбираем возраст из конкретного класса билета и заполняем медианой



In [ ]:
df.isnull().sum() # Проверим сейчас количество пропусков

### Готово, теперь пропусков в данных нет

# Расширенная статистика